11. El Costo del Marketing Inútil (Agregación con Filtro): Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
SELECT 
  COUNT(*) * 1.5 AS costo_perdido_marketing
FROM cursodataacademy.proyecto_hotel.tb_reservas
WHERE distribution_channel = 'TA/TO' AND is_canceled = 1;

12.El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
SELECT 
  COUNT(*) AS reservas_canceladas_tarde,
  COUNT(*) * 120 AS perdida_total_operativa
FROM cursodataacademy.proyecto_hotel.tb_reservas
WHERE is_canceled = 1 
  AND DATEDIFF(
        TO_DATE(CONCAT(CAST(arrival_date_day_of_month AS STRING), ' ', arrival_date_month, ' ', CAST(arrival_date_year AS STRING)), 'd MMMM yyyy'),
        TO_DATE(reservation_status_date, 'd/M/yy')
      ) < 3;

13. (JOIN 1 - LEFT JOIN - Entendiendo el Modelo): Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

In [0]:
-- 1. Crear la tabla auxiliar de países a modo de test
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_paises (
    cod_pais STRING,
    nombre_pais STRING
);

INSERT INTO cursodataacademy.proyecto_hotel.tb_paises VALUES 
('PRT', 'Portugal'),
('GBR', 'Reino Unido'),
('USA', 'Estados Unidos'),
('ESP', 'España'),
('IRL', 'Irlanda'),
('FRA', 'Francia'),
('DEU', 'Alemania'),
('ITA', 'Italia'),
('BRA', 'Brasil'),
('NLD', 'Países Bajos'),
('BEL', 'Bélgica'),
('CHE', 'Suiza'),
('AUT', 'Austria'),
('SWE', 'Suecia'),
(NULL,  'No especificado');

-- 2 Se demuestra la consula con Left Join
SELECT 
  r.is_canceled,
  p.nombre_pais
FROM cursodataacademy.proyecto_hotel.tb_reservas r
LEFT JOIN cursodataacademy.proyecto_hotel.tb_paises p 
  ON r.country = p.cod_pais;

14. (JOIN 2 - INNER JOIN con Agrupación): Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?

In [0]:
SELECT 
  p.nombre_pais,
  COUNT(*) AS total_cancelaciones
FROM cursodataacademy.proyecto_hotel.tb_reservas r
INNER JOIN cursodataacademy.proyecto_hotel.tb_paises p 
  ON r.country = p.cod_pais
WHERE r.is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY total_cancelaciones DESC
LIMIT 5;

15.  (JOIN 3 - Simulación FULL OUTER JOIN): Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.


In [0]:
-- 1. Crear la tabla de test tb_vendedores de RRHH
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_vendedores (
    id_vendedor STRING,
    nombre_vendedor STRING
);

-- 2. Poblamos la tabla de vendedores
INSERT INTO cursodataacademy.proyecto_hotel.tb_vendedores VALUES 
('Direct', 'Carlos Martines'),
('Corporate', 'María Perez'),
('TA/TO', 'Luis Torres'),
('GDS', 'Ana Ramirez'),
('Canal_Retail_Nuevo', 'Sofía Rodriguez'); 

-- 3. Ejecutar el FULL OUTER JOIN entre las reservas y los vendedores
SELECT 
  r.distribution_channel,
  v.nombre_vendedor,
  COUNT(*) AS total_registros
FROM cursodataacademy.proyecto_hotel.tb_reservas r
FULL OUTER JOIN cursodataacademy.proyecto_hotel.tb_vendedores v 
  ON r.distribution_channel = v.id_vendedor
GROUP BY r.distribution_channel, v.nombre_vendedor;

16. (JOIN 4 - RIGHT JOIN): El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).

In [0]:
-- 1. Creamos la tabla de facturas de prueba de test
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_facturas (
    id_reserva INT,
    monto_facturado DECIMAL(10,2)
);

INSERT INTO cursodataacademy.proyecto_hotel.tb_facturas VALUES 
(1, 250.00),
(2, 420.50),
(3, 180.75),
(4, 95.00),
(9999, 600.00); -- para probar el RIGHT JOIN

-- 2. Ejecutamos el RIGHT JOIN 
WITH reservas_con_id AS (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY hotel) AS id_reserva,
        hotel
    FROM cursodataacademy.proyecto_hotel.tb_reservas
)
SELECT 
    r.hotel,
    f.id_reserva,
    f.monto_facturado
FROM reservas_con_id r
RIGHT JOIN cursodataacademy.proyecto_hotel.tb_facturas f 
    ON r.id_reserva = f.id_reserva;